In [2]:
%load_ext autoreload
%autoreload 2

In [3]:

import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
# import pandas_ta as ta
import math
# from tqdm import tqdm
# import gc
# import time
import json
from pprint import pprint
# import pandas_ta
# import talib
import pickle
# from position_tools import calculate_trades, calculate_positions, count_since_last_signal

# from pklibs import *

import logging

# Configure logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

import talib

# from pklib.strategy import *
from pklib.utilities import *
# from pklib.pkindicators import calculate_zigzag
from pklib.indicators import *

# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import confusion_matrix
# from sklearn.metrics import precision_score, recall_score

# import seaborn as sns
# from pklib.rl import *

### IMPORTANT
# pip install numpy==1.26.4 pandas==2.2.1 --force-reinstall

In [4]:
import dotenv
import os

# Reload the variables in your '.env' file (override the existing variables)
dotenv.load_dotenv(".env", override=True)

# 'MY_VAR' is refreshed now
print('HIP_VISIBLE_DEVICES = ', os.environ.get('HIP_VISIBLE_DEVICES')) # MY_VAR = HELLO_BOB

HIP_VISIBLE_DEVICES =  0


In [5]:
import torch
import torch as T
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader
from sklearn.metrics import confusion_matrix, precision_recall_fscore_support
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

from sklearn.metrics import precision_recall_fscore_support


In [8]:

def get_signal(df, lookback_period, target_period, min_target_pct, risk_2_reward, lookback_column='low'):
    fwd_min = df.low.rolling(target_period).min().shift(-target_period)
    fwd_max = df.high.rolling(target_period).max().shift(-target_period)
    fwd_rwd_pct = (fwd_max / df.close) - 1    
    stoploss = df[lookback_column].rolling(lookback_period).min() 
    
    risk_pct = df.close / stoploss - 1
    target_pct = risk_pct * risk_2_reward
    target = df.close * (1 + target_pct)
    
    Y = (fwd_min > stoploss)
    Y = Y & (fwd_rwd_pct > target_pct)
    Y = Y & (target_pct > min_target_pct)
    Y = Y.fillna(False)
    return Y, (stoploss, target), (risk_pct, target_pct) 
    
    


In [20]:
import pandas as pd
import numpy as np
import talib

def calculate_talib_indicator(data, indicator_name, params_list, price_column='close'):
    """
    Calculate a TA-Lib indicator for multiple periods or parameters.

    Parameters:
    - data: pandas DataFrame containing price data.
    - indicator_name: string, name of the TA-Lib indicator function as a string.
    - params_list: list of dictionaries, each containing parameters for the indicator.
    - price_column: string, the column name in 'data' to use for price (default is 'close').

    Returns:
    - result_df: pandas DataFrame with indicator values for each set of parameters.
    """
    # Initialize an empty DataFrame to store results
    result_df = pd.DataFrame(index=data.index)

    # Get the indicator function from TA-Lib
    indicator_func = getattr(talib, indicator_name)

    for params in params_list:
        # Extract parameters
        params_copy = params.copy()  # Copy to avoid modifying the original
        suffix = '_'.join(f"{k}{v}" for k, v in params.items())

        # Prepare input data based on the indicator's requirements
        # Assuming most indicators use the 'close' price, adjust as needed
        input_data = data[price_column]

        # Calculate the indicator
        result = indicator_func(input_data, **params_copy)

        # If the result is a tuple (some indicators return multiple outputs), handle it
        if isinstance(result, tuple):
            for idx, res in enumerate(result):
                result_df[f"{indicator_name}_{suffix}_{idx}"] = res
        else:
            result_df[f"{indicator_name}_{suffix}"] = result

    return result_df

# Load your data
from pklib.utilities import load_candles

exchange = 'binance'
asset = 'AVAX'
quote = 'USDT'
timeframe = '8h'

df = load_candles(exchange, asset, quote, timeframe)

# Define periods for moving averages and standard deviation
periods = [5, 10, 20, 50, 100, 200]

# Calculate Simple Moving Averages (SMA)
sma_df = calculate_talib_indicator(df, 'SMA', [{'timeperiod': p} for p in periods])

# Calculate Exponential Moving Averages (EMA)
ema_df = calculate_talib_indicator(df, 'EMA', [{'timeperiod': p} for p in periods])

# Calculate Standard Deviation
std_df = calculate_talib_indicator(df, 'STDDEV', [{'timeperiod': p, 'nbdev': 1} for p in periods])

# Combine all indicators with the original DataFrame
df = pd.concat([df, sma_df, ema_df, std_df], axis=1)

df

,open,high,low,close,volume,SMA_timeperiod5,SMA_timeperiod10,SMA_timeperiod20,SMA_timeperiod50,SMA_timeperiod100,...,EMA_timeperiod20,EMA_timeperiod50,EMA_timeperiod100,EMA_timeperiod200,STDDEV_timeperiod5_nbdev1,STDDEV_timeperiod10_nbdev1,STDDEV_timeperiod20_nbdev1,STDDEV_timeperiod50_nbdev1,STDDEV_timeperiod100_nbdev1,STDDEV_timeperiod200_nbdev1
timestamp,,,,,,,,,,,,,,,,,,,,,
2020-09-22 00:00:00,0.8500,6.0000,0.8500,4.9096,6369386.27,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-09-22 08:00:00,4.9096,7.0000,4.3110,4.6699,18218967.58,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-09-22 16:00:00,4.6691,5.5499,4.0266,5.3193,6853529.31,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-09-23 00:00:00,5.3279,5.3600,3.9169,4.1197,4289174.65,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2020-09-23 08:00:00,4.1184,4.6172,4.0191,4.1073,4378054.41,4.62516,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,0.466563,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2024-08-22 00:00:00,23.5300,23.6700,23.0500,23.2600,422581.91,23.09000,22.178,21.3645,21.2558,24.1321,...,21.874558,22.019904,23.348120,25.888194,0.437173,1.049617,1.127034,0.866326,3.651204,3.052502
2024-08-22 08:00:00,23.2500,25.3500,23.1900,24.6100,1482070.16,23.56000,22.571,21.5880,21.3572,24.0966,...,22.135076,22.121476,23.373108,25.875476,0.542697,1.146433,1.293038,0.952036,3.629060,3.035236
2024-08-22 16:00:00,24.6100,25.3100,24.0900,25.2200,547083.05,23.94400,23.035,21.8220,21.4410,24.0648,...,22.428878,22.242987,23.409680,25.868954,0.827444,1.185042,1.490586,1.093450,3.605065,3.023781


In [248]:
# df['2017-09-06':]

In [15]:
device = T.device('cuda:0' if T.cuda.is_available() else 'cpu')
class LSTMBinary(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, dropout):
        super(LSTMBinary, self).__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout
        )
        self.fc = nn.Linear(hidden_size, 1)  # Output layer

    def forward(self, x):
        # x shape: (batch_size, window_size, num_features)
        out, _ = self.lstm(x)  # out shape: (batch_size, window_size, hidden_size)
        out = out[:, -1, :]    # Get the output of the last time step (batch_size, hidden_size)
        out = self.fc(out)     # Linear layer (batch_size, 1)
        return out             # Output shape: (batch_size, 1)

# Define the evaluation function
def evaluate_model(model, x_data, y_data):
    model.eval()  # Set the model to evaluation mode
    
    with torch.no_grad():  # Disable gradient computation
        # Make predictions
        outputs = model(x_data.to(device))
        predicted = torch.sigmoid(outputs) > 0.5  # Convert logits to binary predictions
        
        # Flatten predictions and ground truth
        # predicted_flat = predicted.view(-1).cpu().numpy()
        # y_flat = y_data.view(-1).cpu().numpy()
        predicted_flat = predicted[:,-1,:].view(-1).cpu().numpy()
        y_flat = y_data[:,-1,:].view(-1).cpu().numpy()
        
        return predicted_flat, y_flat

def output_performance(predicted_flat, y_flat, r2r, dataset_name="Dataset", do_plot=False):
    # Compute confusion matrix
    cm = confusion_matrix(y_flat, predicted_flat)
    # Compute precision, recall, and F1-score
    precision, recall, f1, _ = precision_recall_fscore_support(y_flat, predicted_flat, average='binary')

    # Display the confusion matrix
    cm_df = pd.DataFrame(cm, index=['Actual 0', 'Actual 1'], columns=['Predicted 0', 'Predicted 1'])
    print(f"\nConfusion Matrix - {dataset_name}:\n{cm_df}")
    
    # Output precision, recall, and F1-score
    print(f"Precision ({dataset_name}): {precision:.4f}")
    print(f"Recall ({dataset_name}): {recall:.4f}")
    print(f"F1-score ({dataset_name}): {f1:.4f}")
    
    # Calculate accuracy
    accuracy = accuracy_score(y_flat, predicted_flat)
    print(f"Accuracy ({dataset_name}): {accuracy:.4f}")
    
    # Calculate cumulative returns
    r = np.where(predicted_flat == 1, np.where(y_flat == 1, r2r, -1), 0)
    cumulative_returns = np.cumsum(r)
    
    if do_plot:
        plt.figure(figsize=(10, 6))
        plt.plot(cumulative_returns)
        plt.title(f"Cumulative Returns - {dataset_name}")
        plt.xlabel("Trade Number")
        plt.ylabel("Cumulative Return")
        plt.grid(True)
        plt.show()
    
    return cm_df, precision, recall, f1, accuracy, cumulative_returns
        # return 
        
        

In [ ]:

# Parameters
num_features = X.shape[1]
input_size = window_size * num_features # x_train.size(2)  # Number of features in X_train
hidden_size = 64   # Number of hidden units in the LSTM
num_layers = window_size                 # Number of LSTM layers
model = LSTMBinary(input_size, hidden_size, num_layers, dropout=0.3)
###########################
num_samples = len(X_train)
batch_size = 1000
num_windows = num_samples - window_size + 1

    # Training loop
num_epochs = 1000
optimizer = optim.Adam(model.parameters(), lr=1e-1)
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=(num_epochs // 100), gamma=0.95)
for epoch in range(num_epochs):
    model.train()
    
    for batch_start in range(0, num_windows, batch_size):
        batch_end = min(batch_start + batch_size, num_windows)    
        for i in range(batch_start, batch_end):
            window = X_train.iloc[i:i + window_size]        
            X_train_windows = create_rolling_windows(window, window_size=window_size)
            # Convert back to a PyTorch tensor
            x_train = torch.tensor(X_train_windows, dtype=torch.float32).to(device)
            y_train = torch.tensor(Y_train[window_size-1:], dtype=torch.float32).unsqueeze(1).repeat(1, window_size).unsqueeze(-1).to(device)
        
            num_neg = (y_train == 0).sum().item()
            num_pos = (y_train == 1).sum().item()

            if num_pos == 0:
                raise ValueError("No positive samples in Y_train, cannot compute pos_weight.")

            pos_weight = torch.tensor([num_neg / num_pos]).to(device)

            # Define the loss function with pos_weight
            criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)

            # Optimizer with a lower learning rate

            # Optional: Learning rate scheduler

            # Normalize input data (if not already normalized)
            # Example using torch
            mean = x_train.mean(dim=(0,1), keepdim=True)
            std = x_train.std(dim=(0,1), keepdim=True)
            x_train = (x_train - mean) / (std + 1e-8)  # Add epsilon to avoid division by zero

            # Zero the gradients
            optimizer.zero_grad()
            
            # Forward pass
            outputs = model(x_train.to(device))
        
            # Check for NaNs or Infs in outputs
            if torch.isnan(outputs).any() or torch.isinf(outputs).any():
                print(f"Epoch {epoch + 1}: Model outputs contain NaNs or Infs. Stopping training.")
                break
            
            # Compute loss
            loss = criterion(outputs, y_train.to(device))
        
            # Check for NaNs in loss
            if torch.isnan(loss):
                print(f"Epoch {epoch + 1}: Loss is NaN. Stopping training.")
                break
            
            # Backward pass
            loss.backward()
            
            # Gradient clipping
            torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            
            # Optimization step
            optimizer.step()
            
            # Scheduler step
            scheduler.step()
            
            if (epoch + 1) % 10 == 0:
                current_lr = optimizer.param_groups[0]['lr']
                print(f'Epoch [{epoch + 1}/{num_epochs}], Loss: {loss.item():.4f}, Learning Rate: {current_lr:.6f}')

# Evaluate the model after training
# predicted_flat_train, y_flat_train = evaluate_model(model, x_train, y_train)
# predicted_flat_test, y_flat_test = evaluate_model(model, x_test, y_test)

# output_performance(predicted_flat_train, y_flat_train, r2r, dataset_name="Train", do_plot=False)
# output_performance(predicted_flat_test, y_flat_test, r2r, dataset_name="Test", do_plot=True)
# r = (predicted_flat * ( (predicted_flat == y_flat)*3 - 1 ) ) * 100
# # r = pd.DataFrame(r, index=df.index[window_size-1:])
# plt.plot(r.cumsum())

In [ ]:
len(outputs), len(x_train)

In [135]:

# torch.save(model.state_dict(), f"models/lstm.pth")

model = RollingWindowLSTMBinaryClassifier(input_size, hidden_size, num_layers, dropout=0.2)
# model.load_state_dict(torch.load(f"models/lstm.pth"))

In [ ]:

predicted_flat_train, y_flat_train = evaluate_model(model, x_train, y_train)
predicted_flat_test, y_flat_test = evaluate_model(model, x_test, y_test)

output_performance(predicted_flat_train, y_flat_train, r2r, dataset_name="Train", do_plot=False)
output_performance(predicted_flat_test, y_flat_test, r2r, dataset_name="Test", do_plot=True)



In [ ]:
exchange = 'binance'; asset = 'ADA'; quote = 'USDT'; test_size=0;resample = None # '6h';
# exchange = 'kucoinfutures/futures'; quote = 'USDT_USDT'
q_r2r = 1; transaction_fee = 0.001; slippage = 0.005;
max_holding_period = lookahead
# Define the array of timeframes
timeframes = ['6h', '8h', '12h', '1d', '3d', '1w']

# Initialize an empty list to collect the results
all_results = []

# Loop through each timeframe, perform backtest, and store results
for timeframe in timeframes:
    # Perform backtest for the current timeframe
    backtest_results = do_backtest(model,exchange, asset, quote, timeframe, resample, 
                             r2r=r2r*q_r2r, 
                             max_holding_period = max_holding_period,
                             do_print=False, 
                             line_label=timeframe, 
                             transaction_fee=transaction_fee, 
                             slippage=slippage)
    
    trades_df = backtest_results['trades_df']
    # Append the results to the all_results list
    all_results.append({
        'timeframe': timeframe,  # Store the timeframe for reference
        'results': backtest_results    # Store the results DataFrame
    })
    trades_df['cum_log_returns'] = (trades_df['exit'] / trades_df['entry']).apply(np.log).cumsum()
    # Calculate cumulative log returns and plot with legend
    # cumulative_log_returns = (results_df['exit'] / results_df['entry']).apply(np.log).cumsum()#.apply(np.expm1)
    trades_df['cum_log_returns'].plot(label=timeframe)  # Add the current timeframe as the label for the legend

# Add the legend to the plot
plt.legend(title='Timeframes')
plt.title('Cumulative Log Returns Across Timeframes')
plt.xlabel('Trades')
plt.ylabel('Cumulative Log Returns')
plt.grid(True)

# Now all_results contains the backtest results for each timeframe


In [ ]:
[(r['timeframe'],len(r['results']['trades_df']), r['results']['metrics']['avg_percentage_return']) for r in all_results]
# [r['results']['metrics']['avg_percentage_return'] for r in all_results]

In [136]:


# exchange = 'kucoinfutures/futures'; asset = 'AVAX'; quote = 'USDT_USDT'; timeframe='8h-futures'; test_size=0;resample = None # '6h';
exchange = 'binance'; asset = 'AVAX'; quote = 'USDT'; timeframe='8h'; test_size=0;resample = None # '6h';
def do_backtest(model, exchange,asset,quote,timeframe,resample, r2r, max_holding_period, do_print=True, do_plot=True, line_label='',transaction_fee=0.001, slippage=0.001):
    df = load_candles_and_indicators(exchange,asset,quote,timeframe,resample)
    df = df.dropna()
    x_train, y_train, x_test, y_test = prepare_ml_features(df, test_size=test_size,window_size=window_size,lookahead=max_holding_period, min_rwd=min_rwd, lookback_period=lookback_period, r2r=r2r, lookback_column='low')

    predicted_flat_train, y_flat_train = evaluate_model(model, x_train, y_train)
    # predicted_flat_test, y_flat_test = evaluate_model(model, x_test, y_test)

    output_performance(predicted_flat_train, y_flat_train, r2r, dataset_name="Train")
    # output_performance(predicted_flat_test, y_flat_test, dataset_name="Test")


    ohlc_data = df.iloc[window_size-1:]
    buy_signals = predicted_flat_train
    stoploss_levels = df['low'].rolling(lookback_period).min()
    max_holding_period = lookahead
    risk_to_reward = r2r
    # transaction_fee=0.001; slippage=0.001;


    backtest_results = backtest(
        ohlc_data, buy_signals, stoploss_levels, 
        max_holding_period, risk_to_reward, 
        transaction_fee=transaction_fee, slippage=slippage
    )
    # results_df = backtest_results['trade_results']
    # if do_plot: (results_df.exit/results_df.entry).apply(np.log).cumsum().plot(legend=line_label); 
    # if do_print: pprint({k:v for k,v in backtest_results.items() if k not in ['trade_results','cumulative_pnl']})
    return backtest_results


In [ ]:
results_df.outcome.value_counts()

In [90]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def backtest(ohlc_data, buy_signals, stoploss_levels, max_holding_period, risk_to_reward, transaction_fee=0, slippage=0):
    results = []
    cumulative_pnl = []

    equity_curve = 0
    max_equity = 0
    drawdown = 0
    total_risk = 0  # To calculate risk-adjusted return

    in_trade = False  # Flag to track if we're currently in a trade
    i = 0  # Initialize index

    while i < len(buy_signals) - 1:
        if buy_signals[i] == 1 and not in_trade:  # Buy signal and no open position
            in_trade = True
            entry_index = i + 1  # Entry happens at the next bar after the buy signal

            if entry_index >= len(ohlc_data):
                break  # Prevent index out of range

            entry_price = ohlc_data['open'][entry_index] * (1 + slippage)  # Buy at next open price with slippage
            stoploss = stoploss_levels[i]
            target_price = entry_price + (entry_price - stoploss) * risk_to_reward  # Calculate target price

            trade_outcome = None
            exit_index = None

            for j in range(entry_index, min(entry_index + max_holding_period, len(ohlc_data))):
                high = ohlc_data['high'][j]
                low = ohlc_data['low'][j]

                # Check if stoploss is hit
                if low <= stoploss:
                    exit_price = stoploss * (1 - slippage)  # Apply slippage to exit price
                    exit_index = j
                    trade_outcome = {'entry': entry_price, 'exit': exit_price, 'outcome': 'Loss', 'holding_period': j - i}
                    break

                # Check if target price is hit
                if high >= target_price:
                    exit_price = target_price * (1 - slippage)  # Apply slippage to exit price
                    exit_index = j
                    trade_outcome = {'entry': entry_price, 'exit': exit_price, 'outcome': 'Win', 'holding_period': j - i}
                    break

            # If max holding period is reached without hitting stoploss or target
            if trade_outcome is None:
                exit_index = min(entry_index + max_holding_period, len(ohlc_data) - 1)
                exit_price = ohlc_data['close'][exit_index] * (1 - slippage)
                trade_outcome = {
                    'entry': entry_price,
                    'exit': exit_price,
                    'outcome': 'Neutral',
                    'holding_period': exit_index - i
                }

            # Record the trade outcome
            trade_outcome['entry_index'] = entry_index
            trade_outcome['exit_index'] = exit_index
            results.append(trade_outcome)

            # Reset in_trade flag and advance i to exit_index to avoid overlapping trades
            in_trade = False
            i = exit_index + 1
        else:
            i += 1  # Move to the next index if no trade is taken or trade is already open

    # Create a DataFrame to store results
    results_df = pd.DataFrame(results)
    if not results_df.empty:
        results_df.set_index(ohlc_data.index[results_df.exit_index], inplace=True)

        # Calculate percentage returns and statistics
        results_df['percentage_return'] = ((results_df['exit'] - results_df['entry']) / results_df['entry']) - 2 * transaction_fee
        total_percentage_return = results_df['percentage_return'].sum()
        num_trades = len(results_df)
        num_wins = (results_df['percentage_return'] >= 0).sum()
        num_losses = (results_df['percentage_return'] < 0).sum()
        win_rate = num_wins / num_trades if num_trades > 0 else 0
        loss_rate = num_losses / num_trades if num_trades > 0 else 0
        avg_percentage_return = results_df['percentage_return'].mean()

        # Sharpe Ratio calculation (simplified, assuming risk-free rate is 0)
        sharpe_ratio = avg_percentage_return / results_df['percentage_return'].std() if num_trades > 0 else 0

        # Calculate Trade Expectancy
        avg_win = results_df[results_df['percentage_return'] > 0]['percentage_return'].mean()
        avg_loss = results_df[results_df['percentage_return'] < 0]['percentage_return'].mean()
        expectancy = (win_rate * avg_win) - (loss_rate * avg_loss)

        # Calculate Risk-Adjusted Return (Total Percentage Return / Total Risk)
        risk_adjusted_return = total_percentage_return / total_risk if total_risk > 0 else 0

        # Calculate Profit Factor
        gross_profit = results_df[results_df['percentage_return'] > 0]['percentage_return'].sum()
        gross_loss = abs(results_df[results_df['percentage_return'] < 0]['percentage_return'].sum())
        profit_factor = gross_profit / gross_loss if gross_loss > 0 else float('inf')

        # Sortino Ratio calculation (focuses only on downside volatility)
        downside_returns = results_df[results_df['percentage_return'] < 0]['percentage_return']
        downside_std = downside_returns.std() if not downside_returns.empty else 0
        sortino_ratio = avg_percentage_return / downside_std if downside_std > 0 else 0

        # Calmar Ratio calculation (average return over max drawdown)
        calmar_ratio = avg_percentage_return / drawdown if drawdown > 0 else float('inf')
    else:
        # If no trades were made, set default values
        total_percentage_return = 0
        num_trades = 0
        num_wins = 0
        num_losses = 0
        win_rate = 0
        loss_rate = 0
        avg_percentage_return = 0
        sharpe_ratio = 0
        avg_win = 0
        avg_loss = 0
        expectancy = 0
        risk_adjusted_return = 0
        gross_profit = 0
        gross_loss = 0
        profit_factor = float('inf')
        sortino_ratio = 0
        calmar_ratio = float('inf')

    # Return results and metrics
    backtest_results = {
        'metrics': {
            'total_percentage_return': total_percentage_return,
            'num_trades': num_trades,
            'num_wins': num_wins,
            'num_losses': num_losses,
            'win_rate': win_rate,
            'loss_rate': loss_rate,
            'avg_percentage_return': avg_percentage_return,
            'sharpe_ratio': sharpe_ratio,
            'sortino_ratio': sortino_ratio,
            'calmar_ratio': calmar_ratio,
            'avg_win': avg_win,
            'avg_loss': avg_loss,
            'expectancy': expectancy,
            'risk_adjusted_return': risk_adjusted_return,
            'gross_profit': gross_profit,
            'gross_loss': gross_loss,
            'profit_factor': profit_factor,
            'max_drawdown': drawdown},
        'trades_df': results_df,
        'cumulative_pnl': cumulative_pnl
    }
    return backtest_results


In [ ]:
# results_df.log_return.cumsum().plot()
(results_df.exit/results_df.entry).apply(np.log).cumsum().plot()

In [ ]:
# (results_df.exit_index - results_df.entry_index).value_counts()
results_df

In [ ]:
list(range(105)[])